In [ ]:
import numpy as np
from astropy.io import fits as fits
from astropy.wcs import WCS
import matplotlib.pyplot as plt

### Use tadpole data as sample:

In [ ]:
hdu = fits.open('/srv/data/chime/tadpole_cutout_Mar2024/canfar/tadpole_freq_cube_gal.fits')
hdr = hdu[0].header
data = hdu[0].data
print(data.shape)
#print(repr(hdr))

### Make frequency array

In [ ]:
freqs = WCS(hdr).all_pix2world(0,0,range(data.shape[1]),0,0)[2]
len(freqs)

### Make sidelobe structure

In [ ]:
def gaussian(x, fwhm, a, b):
    c = fwhm/2.35482
    return a*np.exp(-((x-b)**2)/(2*c**2))

def sidelobes(dx, hQ, fwhmQ, hU, fwhmU, Nlobes, Npix, fwhm):

    xpix = np.arange(0,Npix,1)
    Qvals = np.zeros(Npix)
    Uvals = np.zeros(Npix)

    Qvals = Qvals + gaussian(xpix, fwhm, hQ, Npix/2)
    #Uvals = Uvals + gaussian(xpix, fwhm, hU, Npix/2)
    
    for N in range(0,Nlobes):

        b = Npix/2 + dx*(N+1)
        #Qvals = Qvals - gaussian(xpix, fwhm, gaussian(b, fwhmQ, hQ, Npix/2), b)
        #Uvals = Uvals + gaussian(xpix, fwhm, gaussian(b, fwhmU, hU, Npix/2), b)
        Qvals = Qvals - gaussian(xpix, fwhm, 0.5, b)
        Uvals = Uvals + gaussian(xpix, fwhm, 0.5, b)
        
        b = Npix/2 - dx*(N+1)
        #Qvals = Qvals - gaussian(xpix, fwhm, gaussian(b, fwhmQ, hQ, Npix/2), b)
        #Uvals = Uvals - gaussian(xpix, fwhm, gaussian(b, fwhmU, hU, Npix/2), b)
        Qvals = Qvals - gaussian(xpix, fwhm, 0.5, b)
        Uvals = Uvals - gaussian(xpix, fwhm, 0.5, b)
    
    return xpix, Qvals, Uvals

### Set up frequency-dependent sidelobe kernels

In [ ]:
dx = np.linspace(10,20,len(freqs))
hQ = 1 # height of on-axis Stokes Q leakage - need realistic value
hU = 1 # height of on-axis Stokes U leakage - need realistic value
fwhmQ = np.linspace(45,75,len(freqs)) # defines drop-off of on-axis to off-axis
fwhmU = np.linspace(45,75,len(freqs)) # defines drop-off of on-axis to off-axis
Nlobes = 1 # number of sidelobes to include
Npix = data.shape[2] # number of pixels needed for kernel
fwhm = 5 # width of Gaussian-shaped sidelobes

Qkernel = []
Ukernel = []

for i in range(0,len(dx)):
    
    xpix, Qvals, Uvals = sidelobes(dx[i], hQ, fwhmQ[i], hU, fwhmU[i], Nlobes, Npix, fwhm)
    Qkernel.append(Qvals)
    Ukernel.append(Uvals)

fidx = [0,200,400,600,800]

fig, ax = plt.subplots(1,2,figsize=(12,3))

for i in range(0,len(fidx)):
    ax[0].plot(xpix, Qkernel[fidx[i]], color='C'+str(i), label=f'{freqs[fidx[i]]/1e6} MHz')
    ax[1].plot(xpix, Ukernel[fidx[i]], color='C'+str(i), label=f'{freqs[fidx[i]]/1e6} MHz')
    ax[0].legend(loc='upper right')
    ax[1].legend(loc='upper right')

### Set up an example map (Stokes Q and U the same)

In [ ]:
Npix = data.shape[2]
xpix = np.arange(-(Npix-1)/2,(Npix+1)/2,1)
ypix = np.arange(-(Npix-1)/2,(Npix+1)/2,1)
xgrid,ygrid = np.meshgrid(xpix,ypix)

r = np.sqrt(xgrid**2+ygrid**2)

test_map = np.ones_like(r)*-1.
test_map[r<20] = 1.0

### Convolve with the kernel

In [ ]:
Q_conv_test  = np.convolve(test_map[0], Qkernel[0], mode='same')
wid = len(Q_conv_test)

Q_conv_cube = np.empty([len(freqs),len(ypix),wid])
U_conv_cube = np.empty([len(freqs),len(ypix),wid])

for j in range(0,len(freqs)):
    for i in range(0,len(ypix)):
        Q_conv_cube[j,i] = np.convolve(test_map[i], Qkernel[j], mode='same')
        U_conv_cube[j,i] = np.convolve(test_map[i], Ukernel[j], mode='same')

### Add back the original 'source' map

In [ ]:
Q_cube_mod = np.empty_like(Q_conv_cube)
U_cube_mod = np.empty_like(U_conv_cube)

for j in range(0,len(freqs)):
    Q_cube_mod[j] = test_map + Q_conv_cube[j]*0.1
    U_cube_mod[j] = test_map + U_conv_cube[j]*0.1

fig, ax = plt.subplots(2,4, figsize=(12,5))

ax[0,0].imshow(test_map, origin='lower',cmap='RdBu_r', vmin=-2,vmax=2)
ax[1,0].imshow(test_map, origin='lower',cmap='RdBu_r', vmin=-2,vmax=2)

fidx = [0,400,800]

for i in range(0,3):
    ax[0,i+1].imshow(Q_cube_mod[fidx[i]], origin='lower',cmap='RdBu_r', vmin=-2,vmax=2)
    ax[1,i+1].imshow(U_cube_mod[fidx[i]], origin='lower',cmap='RdBu_r', vmin=-2,vmax=2)
    ax[0,i+1].set_title(f'{freqs[fidx[i]]/1e6} MHz')

### Make intrinsic polarization maps and a Faraday screen

In [ ]:
Npix = data.shape[2]
xpix = np.arange(-(Npix-1)/2,(Npix+1)/2,1)
ypix = np.arange(-(Npix-1)/2,(Npix+1)/2,1)
xgrid,ygrid = np.meshgrid(xpix,ypix)

r = np.sqrt(xgrid**2+ygrid**2)

RM_screen = np.zeros_like(r)
RM_screen[r<20] = 50.0

P0 = np.ones_like(data[0])+np.ones_like(data[0])*1j


In [ ]:
P0.shape

In [ ]:
125+30

### Faraday rotate each frequency channel

In [ ]:
Prot = np.empty_like(P0)

for i in range(0,len(freqs)):
    theta = RM_screen*((2.998e8)/(freqs[i]))**2
    Prot[i] = P0[i]*np.exp(2*theta*1j)

In [ ]:
Q_conv_cube = np.empty([len(freqs),len(ypix),wid])
U_conv_cube = np.empty([len(freqs),len(ypix),wid])

for j in range(0,len(freqs)):
    for i in range(0,len(ypix)):
        Q_conv_cube[j,i] = np.convolve(Prot[j,i].real, Qkernel[j], mode='same')*0.1 + Prot[j,i].real
        U_conv_cube[j,i] = np.convolve(Prot[j,i].imag, Ukernel[j], mode='same')*0.1 + Prot[j,i].imag

In [ ]:
Prot_obs = Q_conv_cube + 1j*U_conv_cube

In [ ]:
from RMtools_1D import do_RMsynth_1D

In [ ]:
dq = np.ones_like(freqs)*0.01
du = np.ones_like(freqs)*0.01

RM_obs = np.empty([P0.shape[1],P0.shape[2]])

#for i in range(0,1):
#    for j in range(0,1):
for i in range(95,155):
    print(i)
    for j in range(95,155):
        #print(i,j)
        data_in = [freqs, Prot_obs[:,i,j].real, Prot_obs[:,i,j].imag,  dq, du]
        mDict, aDict = do_RMsynth_1D.run_rmsynth(data_in,phiMax_radm2=100,dPhi_radm2=1)
        RM_obs[i,j] = mDict['phiPeakPIfit_rm2']
        #print(mDict)
        #print(aDict)

In [ ]:
fig, ax = plt.subplots(1,3,figsize=(12,4))
ax[0].imshow(RM_screen, vmin=-100,vmax=100,origin='lower',cmap='RdBu_r')
ax[1].imshow(RM_obs, vmin=-100,vmax=100,origin='lower',cmap='RdBu_r')
ax[2].imshow(RM_obs-RM_screen, vmin=-1,vmax=1,origin='lower',cmap='RdBu_r')

In [ ]:
RM_obs[0,0]